# Zajęcia 1 — pierwszy eksperyment ML

**Wzorcowe rozwiązanie dla prowadzącego**

Notebook przedstawia kompletne wykonanie mikroinstrukcji z materiału do pierwszych zajęć. Każdy krok znajduje się w osobnej parze komórek: opis oraz implementacja. Notebook należy uruchamiać z katalogu głównego repozytorium.


## 1. Import bibliotek

Importujemy wyłącznie elementy potrzebne do analizy danych, trenowania modeli, ewaluacji, wizualizacji i zapisu artefaktów.


In [1]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


Matplotlib is building the font cache; this may take a moment.


## 2. Katalogi na artefakty

Tworzymy osobne katalogi na raporty i modele. `parents=True` tworzy także brakujące katalogi nadrzędne, a `exist_ok=True` pozwala bezpiecznie ponownie wykonać komórkę.


In [2]:
ARTIFACTS_DIR = Path("artifacts/zajecia_01")
REPORTS_DIR = ARTIFACTS_DIR / "reports"
MODELS_DIR = ARTIFACTS_DIR / "models"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## 3. Wczytanie zbioru Iris

`load_iris(as_frame=True)` zwraca dane w formacie pandas i nie wymaga pobierania osobnego pliku z internetu. Rozdzielamy cechy `X` i etykietę `y`.


In [3]:
iris = load_iris(as_frame=True)
X = iris.data.copy()
y = iris.target.copy()

print(iris.DESCR.splitlines()[0])


.. _iris_dataset:


## 4. Pierwsze wiersze danych

Wyświetlamy próbkę, aby sprawdzić znaczenie kolumn i ogólny format danych.


In [4]:
display(X.head())


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


## 5. Liczba wierszy i kolumn

Sprawdzamy podstawowy rozmiar zbioru.


In [5]:
rows, columns = X.shape
print(f"Liczba wierszy: {rows}")
print(f"Liczba kolumn cech: {columns}")


Liczba wierszy: 150
Liczba kolumn cech: 4


## 6. Nazwy i typy kolumn

Typy danych wpływają na dostępne transformacje oraz sposób walidacji.


In [6]:
column_schema = pd.DataFrame({
    "column": X.columns,
    "dtype": X.dtypes.astype(str).values,
})
display(column_schema)


,column,dtype
0,sepal length (cm),float64
1,sepal width (cm),float64
2,petal length (cm),float64
3,petal width (cm),float64


## 7. Brakujące wartości

Liczymy braki osobno dla każdej cechy.


In [7]:
missing_values = X.isna().sum().rename("missing_values")
display(missing_values.to_frame())


,missing_values
sepal length (cm),0
sepal width (cm),0
petal length (cm),0
petal width (cm),0


## 8. Zduplikowane wiersze

Duplikat nie zawsze jest błędem, ale jego liczba powinna być znana przed treningiem.


In [8]:
duplicate_rows = int(X.duplicated().sum())
print(f"Liczba zduplikowanych wierszy cech: {duplicate_rows}")


Liczba zduplikowanych wierszy cech: 1


## 9. Statystyki opisowe

Podsumowanie pomaga szybko zauważyć zakresy wartości i różnice skali między cechami.


In [9]:
display(X.describe().T)


,count,mean,std,min,25%,50%,75%,max
sepal length (cm),150.0,5.843333,0.828066,4.3,5.1,5.80,6.4,7.9
sepal width (cm),150.0,3.057333,0.435866,2.0,2.8,3.00,3.3,4.4
petal length (cm),150.0,3.758000,1.765298,1.0,1.6,4.35,5.1,6.9
petal width (cm),150.0,1.199333,0.762238,0.1,0.3,1.30,1.8,2.5


## 10. Nazwy i liczebność klas

Zamieniamy numery etykiet na czytelne nazwy gatunków i sprawdzamy rozkład klas.


In [10]:
target_name_map = dict(enumerate(iris.target_names))
y_names = y.map(target_name_map)
class_counts = y_names.value_counts().sort_index()
display(class_counts.rename("count").to_frame())


,count
target,
setosa,50
versicolor,50
virginica,50


## 11. Wykres liczebności klas

Tworzymy pierwszy artefakt graficzny i zapisujemy go jako plik PNG.


In [11]:
ax = class_counts.plot.bar(
    title="Liczebność klas w zbiorze Iris",
    color="#4C78A8",
)
ax.set_xlabel("Klasa")
ax.set_ylabel("Liczba obserwacji")
ax.tick_params(axis="x", rotation=0)
ax.figure.tight_layout()
class_distribution_path = REPORTS_DIR / "class_distribution.png"
ax.figure.savefig(class_distribution_path, dpi=150)
plt.show()
plt.close(ax.figure)
print(f"Zapisano: {class_distribution_path}")


Zapisano: artifacts/zajecia_01/reports/class_distribution.png


/var/folders/r0/18mc62gx7gd4f4f2t45c76cr0000gn/T/ipykernel_90558/1407465663.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12–13. Funkcja walidująca dane

Funkcja realizuje podstawowy kontrakt danych: sprawdza obecność danych, zgodność liczby obserwacji, liczbę i typ cech, braki oraz oczekiwany zbiór klas. Każdy błąd ma czytelny komunikat.


In [12]:
def validate_data(features: pd.DataFrame, target: pd.Series) -> None:
    """Sprawdza podstawowy kontrakt zbioru Iris."""
    if features.empty:
        raise ValueError("Zbiór cech jest pusty.")
    if len(features) != len(target):
        raise ValueError("Liczba wierszy cech i etykiet jest różna.")
    if features.shape[1] != 4:
        raise ValueError("Zbiór powinien zawierać dokładnie cztery cechy.")
    if features.isna().any().any():
        raise ValueError("Cechy zawierają brakujące wartości.")
    if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in features.dtypes):
        raise TypeError("Wszystkie cechy powinny być numeryczne.")
    if set(target.unique()) != {0, 1, 2}:
        raise ValueError("Etykieta zawiera nieoczekiwane klasy.")


## 14. Test funkcji walidującej

Najpierw sprawdzamy poprawne dane. Następnie celowo usuwamy jedną cechę i potwierdzamy, że walidacja odrzuca błędny zbiór.


In [13]:
validate_data(X, y)
print("Poprawne dane przeszły walidację.")

invalid_X = X.drop(columns=[X.columns[-1]])
try:
    validate_data(invalid_X, y)
except ValueError as error:
    print(f"Oczekiwany błąd dla niepoprawnych danych: {error}")
else:
    raise AssertionError("Walidacja powinna odrzucić dane bez jednej kolumny.")


Poprawne dane przeszły walidację.
Oczekiwany błąd dla niepoprawnych danych: Zbiór powinien zawierać dokładnie cztery cechy.


## 15. Podział na dane treningowe i testowe

Wydzielamy 20% danych do końcowej ewaluacji. `stratify=y` zachowuje proporcje klas, a `random_state=42` umożliwia odtworzenie podziału.


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Trening: {X_train.shape}, test: {X_test.shape}")


Trening: (120, 4), test: (30, 4)


## 16. Model bazowy

Klasyfikator bazowy zawsze wybiera najczęstszą klasę. Stanowi minimalny punkt odniesienia dla właściwego modelu.


In [15]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)


## 17. Metryki modelu bazowego

Obliczamy accuracy i macro F1. Macro F1 nadaje każdej klasie taką samą wagę.


In [16]:
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_f1_macro = f1_score(
    y_test,
    baseline_predictions,
    average="macro",
)
print(f"Baseline accuracy: {baseline_accuracy:.4f}")
print(f"Baseline macro F1: {baseline_f1_macro:.4f}")


Baseline accuracy: 0.3333
Baseline macro F1: 0.1667


## 18. Potok modelu właściwego

Łączymy standaryzację i regresję logistyczną w jednym `Pipeline`. Dzięki temu ta sama transformacja zostanie zastosowana podczas treningu i predykcji.


In [17]:
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(max_iter=500, random_state=42),
        ),
    ]
)


## 19. Trening i predykcja

Dopasowujemy cały potok wyłącznie do danych treningowych, a następnie generujemy predykcje dla zbioru testowego.


In [18]:
model.fit(X_train, y_train)
predictions = model.predict(X_test)


## 20. Ewaluacja modelu właściwego

Obliczamy te same metryki co dla baseline oraz raport dla poszczególnych klas.


In [19]:
model_accuracy = accuracy_score(y_test, predictions)
model_f1_macro = f1_score(y_test, predictions, average="macro")

print(f"Model accuracy: {model_accuracy:.4f}")
print(f"Model macro F1: {model_f1_macro:.4f}")
print()
print(
    classification_report(
        y_test,
        predictions,
        target_names=iris.target_names,
    )
)


Model accuracy: 0.9333
Model macro F1: 0.9333

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



## 21. Macierz pomyłek

Wykres pokazuje, które klasy są ze sobą mylone. Zapisujemy go jako drugi artefakt graficzny.


In [20]:
display_object = ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=iris.target_names,
    cmap="Blues",
)
display_object.ax_.set_title("Macierz pomyłek")
display_object.figure_.tight_layout()
confusion_matrix_path = REPORTS_DIR / "confusion_matrix.png"
display_object.figure_.savefig(confusion_matrix_path, dpi=150)
plt.show()
plt.close(display_object.figure_)
print(f"Zapisano: {confusion_matrix_path}")


Zapisano: artifacts/zajecia_01/reports/confusion_matrix.png


/var/folders/r0/18mc62gx7gd4f4f2t45c76cr0000gn/T/ipykernel_90558/3727816458.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 22. Zapis metryk

Konwertujemy wartości na standardowy typ `float` i zapisujemy czytelny plik JSON.


In [21]:
metrics = {
    "baseline_accuracy": float(baseline_accuracy),
    "baseline_f1_macro": float(baseline_f1_macro),
    "model_accuracy": float(model_accuracy),
    "model_f1_macro": float(model_f1_macro),
}

metrics_path = REPORTS_DIR / "metrics.json"
with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print(json.dumps(metrics, indent=2))
print(f"Zapisano: {metrics_path}")


{
  "baseline_accuracy": 0.3333333333333333,
  "baseline_f1_macro": 0.16666666666666666,
  "model_accuracy": 0.9333333333333333,
  "model_f1_macro": 0.9333333333333332
}
Zapisano: artifacts/zajecia_01/reports/metrics.json


## 23. Zapis modelu i metadanych

W jednym artefakcie umieszczamy wytrenowany potok oraz informacje potrzebne do przygotowania prawidłowego wejścia i interpretacji wyjścia.


In [22]:
model_artifact = {
    "model": model,
    "feature_names": list(X.columns),
    "target_names": list(iris.target_names),
}
model_path = MODELS_DIR / "model.joblib"
joblib.dump(model_artifact, model_path)
print(f"Zapisano: {model_path}")


Zapisano: artifacts/zajecia_01/models/model.joblib


## 24. Weryfikacja artefaktów

Sprawdzamy automatycznie, czy każdy wymagany plik istnieje i nie jest pusty.


In [23]:
required_files = [
    REPORTS_DIR / "class_distribution.png",
    REPORTS_DIR / "confusion_matrix.png",
    REPORTS_DIR / "metrics.json",
    MODELS_DIR / "model.joblib",
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Brak pliku: {file_path}")
    if file_path.stat().st_size == 0:
        raise ValueError(f"Plik jest pusty: {file_path}")
    print(f"OK: {file_path} ({file_path.stat().st_size} B)")


OK: artifacts/zajecia_01/reports/class_distribution.png (23375 B)
OK: artifacts/zajecia_01/reports/confusion_matrix.png (31272 B)
OK: artifacts/zajecia_01/reports/metrics.json (169 B)
OK: artifacts/zajecia_01/models/model.joblib (2241 B)


## 25. Podsumowanie

Regresja logistyczna osiąga wyższe macro F1 niż klasyfikator bazowy, dlatego jest lepszym z dwóch porównywanych modeli. Wynik należy interpretować razem z raportem klas i macierzą pomyłek, a nie tylko przez accuracy.

Notebook nie jest jeszcze systemem produkcyjnym. Brakuje między innymi wersjonowania danych i modelu, automatycznych testów, CI, kontraktu predykcji, wdrożenia oraz monitoringu.
